# 03. RAG 성능 평가 (RAGAS)
RAGAS 프레임워크로 RAG 파이프라인 성능을 정량 평가합니다.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from src.generation.rag_chain import MedicalRAGChain
from src.embedding.embedder import MedicalEmbedder
from src.retrieval.vector_store import get_chroma_client, get_or_create_collection
from dotenv import load_dotenv
load_dotenv("../.env")

## 체인 초기화

In [ ]:
client = get_chroma_client()
collection = get_or_create_collection(client)
embedder = MedicalEmbedder()
chain = MedicalRAGChain(collection, embedder)
print("체인 초기화 완료")

## 평가 데이터셋 구성
실제 의료 질문 + 예상 정답(ground truth) 설정

In [ ]:
eval_questions = [
    "What are the treatment options for type 2 diabetes?",
    "What are the side effects of metformin?",
    "How does insulin resistance develop?",
    "What is HbA1c and why is it important?",
    "What lifestyle changes help manage diabetes?",
]

ground_truths = [
    ["Treatment options include metformin, insulin, DPP-4 inhibitors, and lifestyle changes."],
    ["Common side effects of metformin include nausea, diarrhea, and gastrointestinal discomfort."],
    ["Insulin resistance develops when cells fail to respond to insulin, often due to obesity and inactivity."],
    ["HbA1c measures average blood glucose over 2-3 months and is used to diagnose and monitor diabetes."],
    ["Lifestyle changes include diet modification, regular exercise, weight loss, and stress management."],
]

## RAG 결과 수집

In [ ]:
answers = []
contexts = []

for q in eval_questions:
    result = chain.run(q, top_k=5)
    answers.append(result["answer"])
    contexts.append([src["text"] for src in result["sources"]])
    print(f"완료: {q[:50]}...")

print(f"
총 {len(answers)}개 질문 처리 완료")

## RAGAS 평가 실행

In [ ]:
dataset = Dataset.from_dict({
    "question": eval_questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": [g[0] for g in ground_truths],
})

result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)
print(result)

## 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
scores = [result[m] for m in metrics]

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics, scores, color=["#4C72B0","#55A868","#C44E52","#8172B2"])
plt.ylim(0, 1)
plt.title("RAGAS 평가 결과", fontsize=14)
plt.ylabel("Score")
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{score:.3f}", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("ragas_result.png", dpi=150)
plt.show()

## 결과 저장

In [ ]:
df_result = pd.DataFrame({"metric": metrics, "score": scores})
df_result.to_csv("ragas_result.csv", index=False)
print(df_result.to_string(index=False))